# UCB Supply Chain — Medicare Part D Data Pipeline
### Section 1 — Install & Import

In [ ]:
!pip install pandas matplotlib seaborn -q

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os
import json

matplotlib.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.0f}'.format)

print("Done.")

---
### Section 2 — Define AED drug list

In [ ]:
AED_DRUGS = [
    "levetiracetam",
    "brivaracetam",
    "lamotrigine",
    "topiramate",
    "valproic acid",      # Medicare Part D uses valproic acid not valproate
    "carbamazepine",
    "oxcarbazepine",
    "lacosamide",
    "zonisamide",
    "perampanel",
]

print(f"Tracking {len(AED_DRUGS)} AED drugs:")
for d in AED_DRUGS:
    print(f"  - {d}")

---
### Section 3 — Load Medicare Part D CSV

Download the file from:
https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug

Select year 2022 → Download → CSV
Save as `medicare_part_d.csv` in the same folder as this notebook.

In [ ]:
PART_D_FILE = "medicare_part_d.csv"

if not os.path.exists(PART_D_FILE):
    print("File not found. Download from:")
    print("https://data.cms.gov/provider-summary-by-type-of-service/medicare-part-d-prescribers/medicare-part-d-prescribers-by-provider-and-drug")
else:
    print(f"Found {PART_D_FILE} — checking columns...")
    # Read just the header first to confirm column names
    sample = pd.read_csv(PART_D_FILE, nrows=3)
    print(f"Columns: {list(sample.columns)}")
    print()
    print(sample.head(3))

In [ ]:
# Load full file — filtered to AED drugs only
# Processes in 100k row chunks to keep memory low

if os.path.exists(PART_D_FILE):
    print("Loading and filtering to AED drugs...")
    chunks = []
    chunk_count = 0

    for chunk in pd.read_csv(PART_D_FILE, chunksize=100000, low_memory=False):
        chunk_count += 1
        # Filter to AED drugs — case insensitive match
        mask     = chunk["Gnrc_Name"].str.lower().str.strip().isin(AED_DRUGS)
        filtered = chunk[mask]
        if len(filtered) > 0:
            chunks.append(filtered)
        print(f"  Chunk {chunk_count}: kept {len(filtered)} AED rows")

    df_raw = pd.concat(chunks, ignore_index=True)
    print(f"\nTotal AED rows loaded: {len(df_raw):,}")
    print(f"Columns: {list(df_raw.columns)}")
    print()
    print(df_raw.head())

    df_raw.to_csv("aed_medicare_partd.csv", index=False)
    print("\nSaved to aed_medicare_partd.csv")

---
### Section 4 — Explore the raw data structure

In [ ]:
df = pd.read_csv("aed_medicare_partd.csv", low_memory=False)

print(f"Shape: {df.shape}")
print()
print("Column types:")
print(df.dtypes)
print()
print("Sample row:")
print(df.iloc[0])

In [ ]:
# Check drug name coverage — see how they appear in the data
print("Drug names as they appear in the dataset:")
print(df["Gnrc_Name"].value_counts())

In [ ]:
# Check state coverage
print(f"States covered: {df['Prscrbr_State_Abrvtn'].nunique()}")
print()
print(df["Prscrbr_State_Abrvtn"].value_counts().head(15))

In [ ]:
# Check for missing values
print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0])

---
### Section 5 — Aggregate demand by drug and state

Aggregate total claims per drug per state — this is our demand signal.

In [ ]:
# Aggregate by drug and state
df_demand = (
    df.groupby(["Gnrc_Name", "Prscrbr_State_Abrvtn"])
    .agg(
        total_claims      = ("Tot_Clms", "sum"),
        total_patients    = ("Tot_Benes", "sum"),
        total_days_supply = ("Tot_Day_Suply", "sum"),
        total_drug_cost   = ("Tot_Drug_Cst", "sum"),
        provider_count    = ("Prscrbr_NPI", "nunique"),
    )
    .reset_index()
    .rename(columns={
        "Gnrc_Name"           : "drug",
        "Prscrbr_State_Abrvtn": "state",
    })
)

print(f"Aggregated demand shape: {df_demand.shape}")
print()
print(df_demand.head(15))

In [ ]:
# Save aggregated demand
df_demand.to_csv("aed_demand_by_state.csv", index=False)
print(f"Saved {len(df_demand)} rows to aed_demand_by_state.csv")

---
### Section 6 — National demand summary by drug

In [ ]:
# National totals per drug
df_national = (
    df_demand.groupby("drug")
    .agg(
        total_claims      = ("total_claims", "sum"),
        total_patients    = ("total_patients", "sum"),
        total_days_supply = ("total_days_supply", "sum"),
        total_drug_cost   = ("total_drug_cost", "sum"),
        states_covered    = ("state", "nunique"),
    )
    .reset_index()
    .sort_values("total_claims", ascending=False)
)

print("National demand summary — all AEDs:")
print(df_national.to_string(index=False))

In [ ]:
# Bar chart — total claims by drug
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    df_national["drug"],
    df_national["total_claims"] / 1_000_000,
    color="#2E5F8A",
    edgecolor="white",
    width=0.6
)
ax.set_title("Total Medicare Part D Claims by AED — 2022", fontsize=14, pad=12)
ax.set_xlabel("Drug")
ax.set_ylabel("Total claims (millions)")
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, df_national["total_claims"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val/1e6:.1f}M", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig("national_claims_by_drug.png", dpi=150)
plt.show()

---
### Section 7 — Regional demand heatmap (top 20 states)

In [ ]:
# Pivot for heatmap — drug vs state
pivot = df_demand.pivot_table(
    index="drug",
    columns="state",
    values="total_claims",
    aggfunc="sum",
    fill_value=0
)

# Keep top 20 states by total claims
top20_states = df_demand.groupby("state")["total_claims"].sum().nlargest(20).index
pivot_top20  = pivot[top20_states]

# Normalize per row for better visual comparison
pivot_norm = pivot_top20.div(pivot_top20.max(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    pivot_norm,
    cmap="Blues",
    linewidths=0.5,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Relative demand (normalized per drug)"},
    fmt=".1f"
)
ax.set_title("Regional Demand Heatmap — AEDs by State (Top 20 States)", fontsize=13, pad=12)
ax.set_xlabel("State")
ax.set_ylabel("Drug")
plt.tight_layout()
plt.savefig("regional_demand_heatmap.png", dpi=150)
plt.show()

---
### Section 8 — UCB drug focus: levetiracetam and brivaracetam

In [ ]:
# Focus on UCB's own drugs
ucb_drugs  = ["levetiracetam", "brivaracetam"]
df_ucb     = df_demand[df_demand["drug"].isin(ucb_drugs)]

# Top 15 states for each UCB drug
for drug in ucb_drugs:
    top15 = (
        df_ucb[df_ucb["drug"] == drug]
        .nlargest(15, "total_claims")
        [["state", "total_claims", "total_patients", "provider_count"]]
    )
    print(f"\nTop 15 states — {drug}:")
    print(top15.to_string(index=False))

In [ ]:
# Side by side comparison — levetiracetam vs brivaracetam by state
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, drug in zip(axes, ucb_drugs):
    data = (
        df_ucb[df_ucb["drug"] == drug]
        .nlargest(15, "total_claims")
        .sort_values("total_claims")
    )
    ax.barh(data["state"], data["total_claims"] / 1000,
            color="#2E5F8A" if drug == "levetiracetam" else "#1D9E75",
            edgecolor="white")
    ax.set_title(f"{drug.title()} — Top 15 States", fontsize=12)
    ax.set_xlabel("Total claims (thousands)")

plt.suptitle("UCB AED Demand by State — Medicare Part D 2022", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("ucb_drug_demand_by_state.png", dpi=150)
plt.show()

---
### Section 9 — Cost analysis

In [ ]:
# Cost per claim by drug — useful for supply chain value calculation
df_national["cost_per_claim"] = df_national["total_drug_cost"] / df_national["total_claims"]
df_national["cost_per_patient"] = df_national["total_drug_cost"] / df_national["total_patients"]

print("Cost analysis by drug:")
print(df_national[["drug", "total_claims", "total_drug_cost", "cost_per_claim", "cost_per_patient"]]
      .sort_values("cost_per_claim", ascending=False)
      .to_string(index=False))

---
### Section 10 — Save final datasets for forecasting pipeline

In [ ]:
# Save all outputs
df_demand.to_csv("aed_demand_by_state.csv", index=False)
df_national.to_csv("aed_demand_national.csv", index=False)
df_ucb.to_csv("ucb_drug_demand.csv", index=False)

print("Files saved:")
print("  aed_demand_by_state.csv  — demand per drug per state (main dataset)")
print("  aed_demand_national.csv  — national totals per drug")
print("  ucb_drug_demand.csv      — levetiracetam and brivaracetam only")
print()
print("SUMMARY")
print("=" * 50)
print(f"  Drugs tracked     : {df_demand['drug'].nunique()}")
print(f"  States covered    : {df_demand['state'].nunique()}")
print(f"  Total rows        : {len(df_demand):,}")
print(f"  Total claims      : {df_demand['total_claims'].sum():,.0f}")
print(f"  Total patients    : {df_demand['total_patients'].sum():,.0f}")
print(f"  Total drug cost   : ${df_demand['total_drug_cost'].sum():,.0f}")
print()
print("Next step: run the forecasting pipeline notebook")